# 01. Exploratory Data Analysis & Causal Feature Engineering

**Course:** ES 335: Machine Learning (2026)  
**Scope:** Strictly restricted to the **80 Development Engines** (`16,340` cycles). The **20 Held-Out Test Engines** remain sealed until final held-out evaluation.

---

In [ ]:
import sys
from pathlib import Path
from IPython.display import Image, display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(REPO_ROOT))

from src.data import load_config, load_development_data, load_engine_splits
from src.eda import screen_sensors_and_settings, fit_pca_health_index
from src.features import build_preprocessing_pipeline, compare_feature_correlations_across_windows

cfg = load_config()
dev_ids, test_ids, _ = load_engine_splits()
dev_df = load_development_data(rul_cap=cfg['framing']['rul_cap'], alarm_w=cfg['framing']['alarm_w'])
lifetimes = dev_df.groupby('unit_id')['max_cycle'].first()

print(f'Development split: {len(dev_ids)} engines ({len(dev_df):,} rows) | Held-out test: {len(test_ids)} engines')
print(f'Dev engine lifetimes: min={lifetimes.min()}, median={lifetimes.median():.0f}, max={lifetimes.max()} cycles')

## 1. Fleet Lifetime Spread & Degradation Alignment
Engine lifespans span **128 to 362 cycles**, which is why any single fixed replacement cycle $K$ either risks early in-service failures or discards substantial useful life. Aligning sensors by cycles remaining before failure ($t - T_{\mathrm{fail}}$) exposes a consistent degradation onset roughly **125 cycles before failure**, motivating the piecewise RUL cap.

In [ ]:
display(Image(filename=str(REPO_ROOT / 'results/figures/engine_lifetime_distribution.png')))
display(Image(filename=str(REPO_ROOT / 'results/figures/sensor_trajectories_aligned.png')))

## 2. Sensor Variance Screening & Unsupervised 1D PCA Health Index
Fitted strictly on the 80 development engines, 3 fixed sea-level operating settings and 7 constant/near-constant sensors (`sensor_1, 5, 6, 10, 16, 18, 19`) are dropped, leaving **14 informative degradation sensors**. The first principal component (PC1) of these 14 standardized sensors captures ~64% of variance and serves as an unsupervised composite health index.

In [ ]:
summary_df, retained_sensors, dropped_channels = screen_sensors_and_settings(dev_df)
print(f'Retained sensors ({len(retained_sensors)}): {retained_sensors}')
print(f'Dropped channels ({len(dropped_channels)}): {dropped_channels}')

pca_df, pca_info = fit_pca_health_index(dev_df, retained_sensors)
print(f"PC1 explained variance: {pca_info['explained_variance_ratio_pc1']*100:.2f}% | r with capped RUL = {pca_info['corr_pc1_with_rul_capped']:+.4f}")

display(summary_df[['channel', 'physical_description', 'std', 'corr_rul_capped', 'retained', 'status']])
display(Image(filename=str(REPO_ROOT / 'results/figures/sensor_correlation_and_pca_health_index.png')))

## 3. Causal Trailing-Window Feature Pipeline ($k = 30$)
All preprocessing steps (`SensorVarianceFilter` -> `TrailingWindowFeatureTransformer` -> `DataFrameStandardScaler`) live inside a scikit-learn `Pipeline`. Each engine's trailing window $[t - k + 1, t]$ computes smoothed level (`mean`), instability (`std`), and drift rate (`slope`) without any look-ahead or cross-engine bleed, producing **43 features** (`cycle` + 14 × 3 trend statistics).

In [ ]:
window_k = int(cfg['features']['window_k'])
pipe = build_preprocessing_pipeline(window_k=window_k, feature_mode='trailing_trends', scale=True)
X_dev = pipe.fit_transform(dev_df)
print(f'Feature matrix shape (k={window_k}): {X_dev.shape}')

corr_df = compare_feature_correlations_across_windows(dev_df, window_k=window_k)
display(corr_df[['sensor', 'physical_description', 'abs_corr_raw', 'abs_corr_mean', 'abs_corr_slope', 'mean_gain_vs_raw']].head(10))
display(Image(filename=str(REPO_ROOT / 'results/figures/trailing_window_features.png')))